# 02 Data Preprocessing

Preprocessing CRLMSSold*.csv data according to findings from the EDA (01_exploration.ipynb) to create clean CSV and prep for week 4.

## Set Up

Loading the same .csv files used in 01_exploration.ipynb

In [1]:
import pandas as pd
import numpy as np
import glob

pd.set_option('display.max_columns', 100)

Data_dir = '../../data' 

data = sorted(glob.glob(f'{Data_dir}/CRMLSSold*.csv'))

#error check
assert len(data) > 0, "No CRMLSSold*.csv files found."
print(f'Found {len(data)} file(s):')
for f in data:
    print(f'  {f}')

Found 6 file(s):
  ../../data\CRMLSSold202512.csv
  ../../data\CRMLSSold202601.csv
  ../../data\CRMLSSold202602.csv
  ../../data\CRMLSSold202603.csv
  ../../data\CRMLSSold202604.csv
  ../../data\CRMLSSold202605.csv


In [2]:
all_csv = []
raw_total = 0

# Apply filters to all files
for file in data:
    #print(f"Processing file: {file}")
    csv = pd.read_csv(file, low_memory=False)
    raw_total += len(csv)

    # restrict analysis per task doc
    csv = csv[(csv['PropertyType'] == 'Residential') & (csv['PropertySubType'] == 'SingleFamilyResidence')].copy()

    all_csv.append(csv)

df = pd.concat(all_csv, ignore_index=True)

# Reverify total files read
print(f'Total files read: {len(all_csv)}')
# Rows before filtering
print(f'Total rows read pre-filter: {raw_total:,}')
print(f'Total rows read post-filter (restrictions): {len(df):,}')

Total files read: 6
Total rows read pre-filter: 124,404
Total rows read post-filter (restrictions): 61,727


### Convert CloseDate to datetime

Necessary step since deduplication and week 4 task requires validity and consistent real dates.

In [3]:
df['CloseDate'] = pd.to_datetime(df['CloseDate'], errors='coerce')

bad_dates = df['CloseDate'].isna().sum()
print(f'CloseDate parse failures: {bad_dates}')

# drop any date parse fails
if bad_dates > 0:
    df = df.dropna(subset=['CloseDate']).reset_index(drop=True)
    print(f'Dropped {bad_dates} rows with unparseable CloseDate.')

CloseDate parse failures: 0


## Deduplication

- EDA saw 33 duplicated ListingKeys
- Of the 33 duplicates, most pairs share the same ClosePrice, but conflict through CloseDates.
- Some pairs conflict on the ClosePrice

In [4]:
#quantify and inspect the conflict groups before touching anything

pre_dedup = len(df)
# print(f'amount of rows before deduplication: {pre_dedup}')

dup_rows = df[df['ListingKey'].duplicated(keep=False)]
price_nunique = dup_rows.groupby('ListingKey')['ClosePrice'].nunique()

conflict_keys = price_nunique[price_nunique > 1].index
print(f'Total Dupe ListingKeys: {len(price_nunique)}')
print(f'- Agreeing with ClosePrice: {(price_nunique == 1).sum()}')
print(f'- Conflict with ClosePrice: {len(conflict_keys)}')

Total Dupe ListingKeys: 33
- Agreeing with ClosePrice: 24
- Conflict with ClosePrice: 9


In [5]:
# Inspecting the conflicting groups
insp_cols = [col for col in ['ListingKey', 'ClosePrice', 'CloseDate', 'ListPrice']
                if col in df.columns]

display(df[df['ListingKey'].isin(conflict_keys)] .sort_values(['ListingKey', 'CloseDate'])[insp_cols])

,ListingKey,ClosePrice,CloseDate,ListPrice
17366,1118859284,875000.0,2026-01-16,870000.0
26100,1118859284,874400.0,2026-02-06,870000.0
6184,1137564166,425000.0,2025-12-26,419900.0
15845,1137564166,435000.0,2026-01-16,419900.0
14864,1144635053,1578331.0,2026-01-29,1659248.0
24415,1144635053,1714623.0,2026-02-12,1659248.0
48683,1145660298,1200000.0,2026-04-03,1250000.0
61151,1145660298,1000000.0,2026-05-04,1250000.0
11364,1149466652,1230000.0,2026-01-20,965000.0
34794,1149466652,1205900.0,2026-03-19,965000.0


### Dealing with Duplicates (ListingKey)

To deal with the duplicates that are re-reported in later month, we will use the recent version of the MLS as sorted using CloseDate (and ContractStatusChangeDate for verification).

In [6]:
sort_cols = ['ListingKey', 'CloseDate']
if 'ContractStatusChangeDate' in df.columns:
    sort_cols.append('ContractStatusChangeDate')

df = (df.sort_values(sort_cols)
        .drop_duplicates(subset='ListingKey', keep='last')
        .reset_index(drop=True))

print(f'Rows removed by dedup: {pre_dedup - len(df)}')
print(f'Remaining rows: {len(df):,}')
assert df['ListingKey'].is_unique

Rows removed by dedup: 33
Remaining rows: 61,694


### Implausible Values

Values that were flaged from the EDA

| Field | Problem | Count (pre-dedup) |
|---|---|---|
| ClosePrice | min = $1.75 reall small value | 4 |
| ClosePrice | max $796M extreme outlier | 7 |
| LivingArea | 0 size is uninhabitable| 19 |
| BedroomsTotal | value of 0, could be a studio needs clarifying | 29 |
| BathroomsTotalInteger | value of 0, not normal for residences| 24 |
| LotSizeAcres | 0 values, potential unit errors (43,560 = 1 acre in sq ft; max 60,113 acres) | 185 where LotSizeAcres = 0 |

In [7]:
# reverify EDA flags
checks = {
    'LivingArea == 0':            (df['LivingArea'] == 0).sum(),
    'BedroomsTotal == 0':         (df['BedroomsTotal'] == 0).sum(),
    'BathroomsTotalInteger == 0': (df['BathroomsTotalInteger'] == 0).sum(),
    'LotSizeAcres == 0':          (df['LotSizeAcres'] == 0).sum(),
    'ClosePrice < $10,000':       (df['ClosePrice'] < 10_000).sum(),
    'ClosePrice > $100M':         (df['ClosePrice'] > 100_000_000).sum(),
}
for label, n in checks.items():
    print(f'{label:<30} {n:>6,} rows')

LivingArea == 0                    19 rows
BedroomsTotal == 0                 29 rows
BathroomsTotalInteger == 0         24 rows
LotSizeAcres == 0                 185 rows
ClosePrice < $10,000                4 rows
ClosePrice > $100M                  7 rows


 ### i. Zero-value structure fields

 From above, it's difficult to accept that a single-family residence is missing a bedroom, bathroom, and is uninhabitable (LivingArea). These are more likely to be data entry issues rather than true values.

Decicion: Drop zero-value entries of BedroomsTotal, BathroomIntegersTotal, and LivingArea.

In [8]:
n_before = len(df)
print(n_before)

implausible_structure = (
    (df['LivingArea'] == 0) |
    (df['BedroomsTotal'] == 0) |
    (df['BathroomsTotalInteger'] == 0)
)
df = df[~implausible_structure].reset_index(drop=True)

print(f'Rows dropped for zero-value fields: {n_before - len(df):,}')
print(f'Remaining rows: {len(df):,}')

61694
Rows dropped for zero-value fields: 54
Remaining rows: 61,640


### ii. ClosePrice: Establishing boundaries

Some of the ClosePrice outliers can only be removed due to the impact on the rest of the distribution. The following code investigates the flagged entries each potential issue causing the flags:
- unit calculation errors
- abiguity

In [9]:
# Inspect the outlier rows
MIN = 10_000
MAX = 100_000_000

out_of_bounds = ~df['ClosePrice'].between(MIN, MAX)

inspect_cols = [col for col in ['ListingKey', 'ClosePrice', 'ListPrice',
                                'LivingArea', 'CountyOrParish', 'City', 'CloseDate']
                if col in df.columns]
inspect = df.loc[out_of_bounds, inspect_cols].copy()
inspect['Close/List'] = (inspect['ClosePrice'] / inspect['ListPrice']).round(4)

print(f'Rows outside [${MIN:,}, ${MAX:,}]: {out_of_bounds.sum()}')
display(inspect.sort_values('ClosePrice'))

Rows outside [$10,000, $100,000,000]: 11


,ListingKey,ClosePrice,ListPrice,LivingArea,CountyOrParish,City,CloseDate,Close/List
13833,1144223437,1.750000e+00,1749000.0,3513.0,Riverside,Palm Desert,2026-01-30,0.0000
34756,1151459976,6.850000e+02,913655.0,2980.0,Contra Costa,Oakley,2026-04-29,0.0007
24423,1147508351,8.000000e+03,8000.0,984.0,San Bernardino,Trona,2026-02-04,1.0000
684,1110035161,8.300000e+03,2000000.0,3886.0,Los Angeles,Altadena,2026-01-17,0.0042
8101,1135757473,4.720000e+08,479000.0,1910.0,San Bernardino,Victorville,2026-01-19,985.3862
23155,1147256230,6.150000e+08,625000.0,2065.0,Riverside,Lake Elsinore,2026-02-27,984.0000
44181,1153305774,6.450000e+08,645000.0,1104.0,San Diego,Oceanside,2026-04-22,1000.0000
25464,1148201765,6.642500e+08,750000.0,1992.0,San Diego,Chula Vista,2026-03-26,885.6667
33462,1151240000,6.990000e+08,699000.0,2114.0,San Diego,Ramona,2026-02-24,1000.0000
30745,1150712418,7.685000e+08,750000.0,1444.0,San Diego,San Diego,2026-04-09,1024.6667


In [ ]:
# Potential unit corrections

FACTORS = [1, 1e-3, 1e3, 1e6]
RATIO = (0.80, 1.25) 

out_idx = df.index[~df['ClosePrice'].between(MIN, MAX)]
df['ClosePrice_repaired'] = 0

repaired, kept_asis, dropped = [], [], []
for i in out_idx:
    price = df.at[i, 'ClosePrice']
    lst   = df.at[i, 'ListPrice']
    if pd.isna(lst) or lst <= 0: #doesn't have a valid comparing price so drop
        dropped.append(i)
        continue

    matches = []
    for f in FACTORS:
        corrected = price * f
        ratio_ok  = RATIO[0] <= corrected / lst <= RATIO[1]
        bounds_ok = (f == 1) or (MIN <= corrected <= MAX)
        if ratio_ok and bounds_ok:
            matches.append(f)

    if matches == [1]:
        kept_asis.append(i) #price is already in bounds and ratio is okay
    elif len(matches) == 1:
        df.at[i, 'ClosePrice'] = price * matches[0]
        df.at[i, 'ClosePrice_repaired'] = 1
        repaired.append((i, price, price * matches[0]))
    else: # IS ABIGUOUS CASE
        dropped.append(i)
    
print(f'Out-of-bounds rows examined: {len(out_idx)}')
print(f'  repaired (unique unit decoding):     {len(repaired)}')
for i, old, new in repaired:
    print(f' ListingKey{i}: ${old:,.2f} -> ${new:,.0f}')
print(f'  kept as-is (real sale, ratio ~1):    {len(kept_asis)}')
print(f'  dropped (ambiguous):                 {len(dropped)}')

n_before = len(df)
df = df.drop(index=dropped).reset_index(drop=True)
print(f'\nRows dropped: {n_before - len(df)}   Remaining: {len(df):,}')

Out-of-bounds rows examined: 11
  repaired (unique unit decoding):     8
    idx 8101: $472,000,000.00 -> $472,000
    idx 13833: $1.75 -> $1,750,000
    idx 23155: $615,000,000.00 -> $615,000
    idx 25464: $664,250,000.00 -> $664,250
    idx 27322: $796,000,000.00 -> $796,000
    idx 30745: $768,500,000.00 -> $768,500
    idx 33462: $699,000,000.00 -> $699,000
    idx 44181: $645,000,000.00 -> $645,000
  kept as-is (real sale, ratio ~1):    1
  dropped (ambiguous):                 2

Rows dropped: 2   Remaining: 61,638


## Missing Values

Handling missing values as found in 01_exploration.ipynb
- decide whether to drop, impute, or flag

## Encoding

Converting categorical fields to numeric

## Normalizing

as needed (TBD)

## Train/Test Split
use the most recent month of available data as the test set, and the X months immediately preceding it as the training set. X is not fixed, treat the training window length as a tunable choice and experiment to determine the optimal value of X.

# Summary
